# 🌳 Pandas en Google Colab · Parte 4 — Modelos de predicción con árboles

**Unidad 7: árbol de decisión, Random Forest, XGBoost y forecasting.** Basado en los ejemplos `16` a `19` del repositorio **CursoPython**.

Dos preguntas sobre los mismos 150 estudiantes de la Parte 3:

1. **Clasificación — alerta temprana:** *¿reprobará este estudiante?*
2. **Forecasting — temporalidad:** *¿qué nota sacará en el control de la próxima semana?*

| Sección | Tema | Ejemplo del repo |
|---|---|---|
| 1 | Árbol de decisión y sobreajuste | `16_arbol_decision.py` |
| 2 | Random Forest e importancia de variables | `17_random_forest.py` |
| 3 | XGBoost: alerta temprana en la semana 6 | `18_xgboost_alerta_temprana.py` |
| 4 | Forecasting semanal con rezagos y división temporal | `19_forecasting_semanal.py` |
| — | Ejercicios guiados + soluciones | `ejercicios_parte4.py` |

Colab ya trae instalados `scikit-learn`, `xgboost` y `matplotlib`.

## 0. Cargar los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, mean_absolute_error

BASE = ("https://raw.githubusercontent.com/hpizarrog-industry40/CursoPython/"
        "main/Aprendiendo_Pandas/datos/")
rendimiento = pd.read_csv(BASE + "rendimiento.csv")           # 1 fila por estudiante
semanal = pd.read_csv(BASE + "seguimiento_semanal.csv")       # 1 fila por estudiante y semana
print("rendimiento:", rendimiento.shape, "| seguimiento_semanal:", semanal.shape)
semanal.head()

`seguimiento_semanal.csv` registra las **16 semanas** del semestre (certámenes en las semanas **8 y 16**):

| Columna | Descripción |
|---|---|
| `semana`, `fecha` | Semana del semestre (1 a 16) y fecha de inicio |
| `horas_estudio` | Horas de estudio de esa semana |
| `asistencia` | Porcentaje de asistencia de esa semana |
| `entrega_tarea` | 1 si entregó la tarea semanal |
| `nota_control` | Nota del control semanal (1–10) — **variable a pronosticar** |

---
## 1. Árbol de decisión
Pasamos de **predecir un número** (regresión) a **predecir una categoría** (clasificación): `reprueba = 1` si `nota_final < 5`.
Un árbol aprende reglas *si … entonces …* fáciles de explicar.

In [ ]:
df = rendimiento.copy()
df["reprueba"] = (df["nota_final"] < 5).astype(int)
print("Reprueban:", df["reprueba"].sum(), "de", len(df))

# horas_sueno se excluye: en la unidad 6 vimos que no aporta
X = pd.get_dummies(df[["horas_estudio", "asistencia", "trabaja", "curso"]],
                   columns=["trabaja", "curso"], drop_first=True, dtype=int)
y = df["reprueba"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(export_text(arbol, feature_names=list(X.columns)))

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(arbol, feature_names=list(X.columns), class_names=["aprueba", "reprueba"],
          filled=True, rounded=True, fontsize=9)
plt.show()

In [ ]:
pred = arbol.predict(X_test)
print(f"Exactitud: {accuracy_score(y_test, pred):.2f}   Recall (reprobados detectados): {recall_score(y_test, pred):.2f}")
pd.DataFrame(confusion_matrix(y_test, pred), index=["real aprueba", "real reprueba"],
             columns=["pred aprueba", "pred reprueba"])

**Exactitud** = aciertos / total. **Recall** = de los que *realmente* reprueban, ¿a cuántos detecta?
En una alerta temprana importa más el recall: es peor **no avisar** a quien va a reprobar.

### Sobreajuste: ¿y si el árbol crece sin límite?

In [ ]:
filas = []
for p in [1, 2, 3, 4, 6, None]:
    a = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_train, y_train)
    filas.append({"profundidad": str(p), "entrenamiento": accuracy_score(y_train, a.predict(X_train)),
                  "prueba": accuracy_score(y_test, a.predict(X_test))})
curva = pd.DataFrame(filas).set_index("profundidad").round(2)
curva

In [ ]:
curva.plot(marker="o", title="Exactitud según la profundidad del árbol"); plt.show()

Sin límite el árbol **memoriza** el entrenamiento (100 %) pero empeora con datos nuevos: eso es **sobreajuste**. Profundidad 3 es el mejor equilibrio aquí.

---
## 2. Random Forest: muchos árboles votando
Cada árbol se entrena con una muestra al azar de estudiantes y de variables; la predicción final es el **voto** de todos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
p_rf = bosque.predict(X_test)
print(f"Random Forest -> exactitud {accuracy_score(y_test, p_rf):.2f}, recall {recall_score(y_test, p_rf):.2f}")

### Importancia de variables: cuidado con la interpretación
Agregamos `horas_sueno` **a propósito** (sabemos que no aporta) para comparar dos medidas de importancia.

In [ ]:
from sklearn.inspection import permutation_importance

d2 = df.assign(horas_sueno=df["horas_sueno"].fillna(df["horas_sueno"].median()))
X2 = pd.get_dummies(d2[["horas_estudio", "asistencia", "horas_sueno", "trabaja", "curso"]],
                    columns=["trabaja", "curso"], drop_first=True, dtype=int)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.3, random_state=42, stratify=y)
bosque2 = RandomForestClassifier(n_estimators=300, random_state=42).fit(X2_train, y2_train)

pd.DataFrame({
    "impureza": bosque2.feature_importances_,
    "permutacion": permutation_importance(bosque2, X2_test, y2_test, n_repeats=30, random_state=42).importances_mean,
}, index=X2.columns).round(3).sort_values("permutacion", ascending=False)

- La **impureza** le da peso a `horas_sueno` (favorece variables numéricas con muchos valores).
- La **permutación** lo deja en ~0: desordenarla no empeora el modelo. ✅ Preferir permutación para interpretar.
- `asistencia` también queda baja porque está correlacionada con `horas_estudio` (r = 0,62): **importancia baja ≠ irrelevante**.

### Comparación honesta con la regresión lineal (Parte 3)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, df["nota_final"], test_size=0.2, random_state=42)
lin = LinearRegression().fit(Xr_train, yr_train)
rfr = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42).fit(Xr_train, yr_train)
print(f"MAE regresión lineal: {mean_absolute_error(yr_test, lin.predict(Xr_test)):.2f}")
print(f"MAE Random Forest   : {mean_absolute_error(yr_test, rfr.predict(Xr_test)):.2f}")

Cuando la relación real es **lineal**, la regresión lineal gana. Los árboles brillan con **umbrales e interacciones** (sección 4).

---
## 3. XGBoost: alerta temprana en la semana 6
XGBoost construye árboles **en serie**: cada uno corrige los errores de los anteriores.

**Pregunta:** con lo que sabemos en la semana 6, ¿quién reprobará? Usamos `groupby` (Unidad 4) para pasar de *1 fila por semana* a *1 fila por estudiante*.

In [ ]:
primeras = semanal[semanal["semana"] <= 6]
resumen = primeras.groupby("id", as_index=False).agg(
    asistencia_prom=("asistencia", "mean"), horas_prom=("horas_estudio", "mean"),
    entregas=("entrega_tarea", "sum"), nota_control_prom=("nota_control", "mean"))
s13 = primeras[primeras["semana"] <= 3].groupby("id")["asistencia"].mean()
s46 = primeras[primeras["semana"] >= 4].groupby("id")["asistencia"].mean()
resumen["tendencia_asistencia"] = (s46 - s13).values

alerta_df = resumen.merge(df[["id", "trabaja", "curso", "reprueba"]], on="id", validate="one_to_one")
print(alerta_df.shape)
alerta_df.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

XA = pd.get_dummies(alerta_df.drop(columns=["id", "reprueba"]), columns=["trabaja", "curso"], drop_first=True, dtype=int)
yA = alerta_df["reprueba"]
XA_train, XA_test, yA_train, yA_test = train_test_split(XA, yA, test_size=0.3, random_state=42, stratify=yA)

modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Árbol (prof. 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42),
}
filas = []
for nombre, m in modelos.items():
    m.fit(XA_train, yA_train)
    p = m.predict(XA_test)
    filas.append({"modelo": nombre, "exactitud": accuracy_score(yA_test, p),
                  "recall": recall_score(yA_test, p),
                  "AUC": roc_auc_score(yA_test, m.predict_proba(XA_test)[:, 1])})
pd.DataFrame(filas).set_index("modelo").round(3)

- Con datos de la **semana 6** ya se detecta a más del 90 % de quienes reprobarán: hay 10 semanas para intervenir.
- **AUC** mide la capacidad de ordenar del más al menos riesgoso (1 = perfecto, 0,5 = azar).
- La regresión logística compite muy bien: un modelo simple puede bastar. Siempre comparar.

### Hiperparámetros principales de XGBoost

In [ ]:
for lr in [0.01, 0.1, 0.5]:
    m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=lr, random_state=42).fit(XA_train, yA_train)
    print(f"learning_rate={lr:<5} AUC = {roc_auc_score(yA_test, m.predict_proba(XA_test)[:, 1]):.3f}")

`n_estimators`: cuántos árboles · `max_depth`: tamaño de cada árbol · `learning_rate`: cuánto corrige cada árbol.

In [ ]:
xgb = modelos["XGBoost"]
lista = alerta_df.loc[XA_test.index, ["id", "asistencia_prom", "nota_control_prom", "reprueba"]].copy()
lista["prob_reprobar"] = xgb.predict_proba(XA_test)[:, 1].round(2)
lista.sort_values("prob_reprobar", ascending=False).head(10)

---
## 4. Forecasting: pronosticar el próximo control semanal
**Pregunta:** con lo que sabemos *hasta la semana anterior*, ¿qué nota sacará cada estudiante esta semana?

### 4.1 Mirar la serie de tiempo

In [ ]:
fc = semanal.merge(rendimiento[["id", "trabaja"]], on="id", validate="many_to_one")
fc = fc.sort_values(["id", "semana"]).reset_index(drop=True)
por_semana = fc.pivot_table(index="semana", columns="trabaja", values=["asistencia", "nota_control"], aggfunc="mean")

fig, (e1, e2) = plt.subplots(1, 2, figsize=(13, 4))
por_semana["asistencia"].plot(ax=e1, marker="o", title="Asistencia promedio por semana")
por_semana["nota_control"].plot(ax=e2, marker="o", title="Nota de control promedio por semana")
for e in (e1, e2):
    for s in (8, 16): e.axvline(s, color="gray", linestyle="--", linewidth=1)
plt.tight_layout(); plt.show()

**Patrones:** la asistencia de quienes trabajan cae desde la semana 7 (**tendencia**), y en los certámenes (semanas 8 y 16)
las notas de ambos grupos se separan (**estacionalidad + interacción**).

### 4.2 Variables con información del pasado: rezagos y medias móviles

In [ ]:
g = fc.groupby("id")
fc["nota_lag1"] = g["nota_control"].shift(1)
fc["nota_lag2"] = g["nota_control"].shift(2)
fc["nota_media3"] = g["nota_control"].transform(lambda s: s.shift(1).rolling(3).mean())
fc["nota_media_acum"] = g["nota_control"].transform(lambda s: s.shift(1).expanding().mean())
fc["asistencia_lag1"] = g["asistencia"].shift(1)
fc["horas_lag1"] = g["horas_estudio"].shift(1)
fc["es_certamen"] = fc["semana"].isin([8, 16]).astype(int)
fc["semanas_para_certamen"] = fc["semana"].apply(lambda s: 8 - s if s <= 8 else 16 - s)
fc["trabaja_si"] = (fc["trabaja"] == "Si").astype(int)
fc[["id", "semana", "nota_control", "nota_lag1", "nota_lag2", "nota_media3"]].head(6)

⚠️ `shift(1)` dentro de `groupby("id")`: cada estudiante usa **su propia** historia y nunca la nota de la semana que queremos pronosticar.

### 4.3 División temporal: entrenar con el pasado, probar con el futuro

In [ ]:
BASE_VARS = ["semana", "es_certamen", "trabaja_si", "nota_lag1", "nota_lag2",
             "nota_media3", "nota_media_acum", "asistencia_lag1", "horas_lag1"]
CAL_VARS = ["semanas_para_certamen"] + BASE_VARS[1:]

datos_fc = fc.dropna(subset=BASE_VARS)
train, test = datos_fc[datos_fc["semana"] <= 12], datos_fc[datos_fc["semana"] >= 13]
print(f"Entrenamiento: semanas 4-12 ({len(train)} filas) | Prueba: semanas 13-16 ({len(test)} filas)")

### 4.4 Líneas base y modelos

In [ ]:
from xgboost import XGBRegressor

def evaluar(variables):
    res = {}
    for nombre, m in {"Regresión lineal": LinearRegression(),
                      "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42),
                      "XGBoost": XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)}.items():
        m.fit(train[variables], train["nota_control"])
        res[nombre] = pd.Series(m.predict(test[variables]), index=test.index)
    return res

pred_base = evaluar(BASE_VARS)
pred_base["Ingenuo (última nota)"] = test["nota_lag1"]
pred_base["Media 3 semanas"] = test["nota_media3"]
cert = test["es_certamen"] == 1
pd.DataFrame({"MAE_total": {n: mean_absolute_error(test["nota_control"], p) for n, p in pred_base.items()},
              "MAE_certamen": {n: mean_absolute_error(test.loc[cert, "nota_control"], p[cert]) for n, p in pred_base.items()}
             }).round(3).sort_values("MAE_total")

XGBoost gana: descubre solo la **interacción** *trabajar × certamen* y el **umbral** *asistencia baja → nota baja la semana siguiente*. La regresión lineal queda incluso detrás del pronóstico ingenuo.

### 4.5 Mejorar las variables antes que el algoritmo

In [ ]:
pred_cal = evaluar(CAL_VARS)
pd.DataFrame({"MAE con 'semana'": {n: mean_absolute_error(test["nota_control"], pred_base[n]) for n in pred_cal},
              "MAE con 'semanas_para_certamen'": {n: mean_absolute_error(test["nota_control"], p) for n, p in pred_cal.items()}
             }).round(3)

Una variable de calendario mejor ayuda a **todos** los modelos. La regresión lineal mejora más con ella que cambiando de algoritmo.

In [ ]:
test = test.assign(pronostico=pred_cal["XGBoost"].clip(1, 10).round(1))
curso = test.groupby("semana")[["nota_control", "pronostico"]].mean().round(2)
curso.plot(marker="o", title="Promedio del curso: real vs. pronóstico XGBoost"); plt.show()
curso

---
## ✍️ Ejercicios guiados (`ejercicios_parte4.py`)
Cada función trae una **guía**. Reemplaza `pass` y ejecuta la celda de comprobación.
Los ejercicios 2 y 3 usan el resultado del 1; el 5 usa el del 4.

In [ ]:
def ejercicio_1_crear_objetivo(df):
    """COPIA de df con columna 'reprueba' = 1 si nota_final < 5, si no 0 (enteros).
    Guía: df = df.copy(); (df["nota_final"] < 5).astype(int)"""
    pass  # TODO

def ejercicio_2_entrenar_arbol(X_train, y_train):
    """Árbol de decisión con max_depth=3 y random_state=42, entrenado. Devuelve el modelo."""
    pass  # TODO

def ejercicio_3_random_forest(X_train, y_train):
    """RandomForestClassifier con n_estimators=200 y random_state=42, entrenado."""
    pass  # TODO

def ejercicio_4_crear_rezagos(semanal):
    """COPIA ordenada por id y semana con 'nota_lag1' y 'nota_media3' (3 semanas anteriores).
    Guía: g = df.groupby("id")["nota_control"]; g.shift(1); g.transform(lambda s: s.shift(1).rolling(3).mean())"""
    pass  # TODO

def ejercicio_5_forecast_xgboost(df, variables):
    """XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42) con
    train = semanas <= 12 y test = semanas >= 13 (tras dropna). Devuelve el MAE en test."""
    pass  # TODO

In [ ]:
# Preparación y comprobación automática (no modificar)
def preparar_clasificacion():
    d = ejercicio_1_crear_objetivo(rendimiento)
    X_ = pd.get_dummies(d[["horas_estudio", "asistencia", "trabaja", "curso"]],
                        columns=["trabaja", "curso"], drop_first=True, dtype=int)
    return train_test_split(X_, d["reprueba"], test_size=0.3, random_state=42, stratify=d["reprueba"])

def preparar_forecasting():
    d = ejercicio_4_crear_rezagos(semanal).merge(rendimiento[["id", "trabaja"]], on="id", how="left")
    d["trabaja_si"] = (d["trabaja"] == "Si").astype(int)
    d["es_certamen"] = d["semana"].isin([8, 16]).astype(int)
    d["semanas_para_certamen"] = d["semana"].apply(lambda s: 8 - s if s <= 8 else 16 - s)
    return d, ["semanas_para_certamen", "es_certamen", "trabaja_si", "nota_lag1", "nota_media3"]

def test_1():
    r = ejercicio_1_crear_objetivo(rendimiento)
    assert "reprueba" not in rendimiento.columns and int(r["reprueba"].sum()) == 56
def test_2():
    a, b, c, d = preparar_clasificacion(); m = ejercicio_2_entrenar_arbol(a, c)
    assert m.get_depth() <= 3 and abs(accuracy_score(d, m.predict(b)) - 0.867) < 0.01
def test_3():
    a, b, c, d = preparar_clasificacion(); m = ejercicio_3_random_forest(a, c)
    assert len(m.estimators_) == 200 and abs(accuracy_score(d, m.predict(b)) - 0.889) < 0.01
def test_4():
    r = ejercicio_4_crear_rezagos(semanal); f = r[(r["id"] == 1) & (r["semana"] == 4)].iloc[0]
    assert f["nota_lag1"] == 10.0 and abs(f["nota_media3"] - 9.633) < 0.01
    assert r[r["semana"] == 1]["nota_lag1"].isna().all()
def test_5():
    d, v = preparar_forecasting(); mae = ejercicio_5_forecast_xgboost(d, v)
    assert abs(mae - 0.629) < 0.01 and mae < 0.743

def comprobar():
    for n, t in enumerate([test_1, test_2, test_3, test_4, test_5], start=1):
        try:
            t(); print(f"Ejercicio {n}: ✅ OK")
        except Exception as e:
            print(f"Ejercicio {n}: ⏳ pendiente o incorrecto ({type(e).__name__})")

comprobar()

### 📝 Interpretación (completa esta celda)
- El árbol detecta al … % de los estudiantes que reprueban (recall). La primera pregunta del árbol es sobre …
- En la alerta temprana prefiero el modelo … porque …
- En el forecasting, XGBoost le gana al pronóstico ingenuo por … puntos de MAE. Dividí los datos por semanas porque …

---
## ✅ Soluciones (revisión en clase)
> 💡 Colapsa este título para ocultar las soluciones hasta la revisión.

In [ ]:
def ejercicio_1_crear_objetivo(df):
    df = df.copy()
    df["reprueba"] = (df["nota_final"] < 5).astype(int)
    return df

def ejercicio_2_entrenar_arbol(X_train, y_train):
    return DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

def ejercicio_3_random_forest(X_train, y_train):
    return RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

def ejercicio_4_crear_rezagos(semanal):
    df = semanal.sort_values(["id", "semana"]).copy()
    g = df.groupby("id")["nota_control"]
    df["nota_lag1"] = g.shift(1)
    df["nota_media3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
    return df

def ejercicio_5_forecast_xgboost(df, variables):
    d = df.dropna(subset=variables)
    train_, test_ = d[d["semana"] <= 12], d[d["semana"] >= 13]
    m = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
    m.fit(train_[variables], train_["nota_control"])
    return mean_absolute_error(test_["nota_control"], m.predict(test_[variables]))

comprobar()

**Resultados esperados:** (1) 56 reprobados · (2) árbol: exactitud 0,87 · (3) Random Forest: exactitud 0,89, recall 0,94 ·
(4) estudiante 1, semana 4: lag1 = 10,0 y media3 = 9,63 · (5) MAE = 0,63 (ingenuo: 0,74).

**Interpretación:** el árbol detecta 88 % de los reprobados y su primera pregunta es sobre las horas de estudio. Random Forest
detecta 94 %. La división **por semanas** evita usar el futuro para predecir el pasado: en producción solo conoceremos la historia.

---
### 🎓 Fin de la Parte 4: de explicar con una recta a anticipar con árboles.